# S18 · Payoff diagrams and put-call parity

Remember the ₹1 lakh booking amount that locked the flat's price? An option is exactly
that kind of deal, written on a share. All through this session we follow **one real
option**: the right to buy Reliance shares at **₹1,560** at any time up to one month
from 31 December 2025, when Reliance closed at about ₹1,563.

In this notebook we draw what that option pays at the end, see the buyer's and the
writer's sides, build three classic strategies by adding payoffs together, and check
put-call parity with two portfolios. No downloads, no formulas beyond `max`.


**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to the words (call, put, strike, premium)? Each one is explained the moment it
  appears, and all of them are in `primers/glossary.md`.
- New to the bell curve? `primers/distributions_and_the_bell_curve.md` is a ten-minute,
  picture-first read.
- Already confident with code or with finance? Look for the cells marked
  **Stretch (optional)** near the end.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook uses only numpy and matplotlib, which Colab already ships.
# So there is nothing to install here.
print("Setup complete - nothing to install.")

## Step 1 — get our tools ready

NumPy does maths on whole lists of numbers at once, and matplotlib draws charts.

In [ ]:
import numpy as np                 # fast maths on lists of numbers
import matplotlib.pyplot as plt    # charts

print("Tools ready.")

## Step 2 — our option, in numbers

The words we need, each tied to our Reliance example:

| Word | Plain meaning | Our example |
|---|---|---|
| **call** | the right to BUY at a fixed price | the right to buy Reliance at ₹1,560 |
| **put** | the right to SELL at a fixed price | the right to sell Reliance at ₹1,560 |
| **strike** `K` | the fixed price in the contract | ₹1,560 |
| **expiry** `T` | the deadline | one month |
| **premium** | what the buyer pays up front | about ₹41 for the call, ₹30 for the put |

The premiums are rounded here. In notebook 2 we compute them exactly with
Black-Scholes: ₹40.81 and ₹29.86.

In [ ]:
K = 1560                 # strike, in rupees
S_today = 1563.17        # Reliance's price on 31 Dec 2025
call_premium = 41        # what the call buyer pays (rounded)
put_premium = 30         # what the put buyer pays (rounded)

# 200 possible prices for Reliance on expiry day, from 1,350 to 1,770.
S_T = np.linspace(1350, 1770, 200)

print("strike:", K, "  price today:", S_today)
print("we will look at expiry prices from", S_T.min(), "to", S_T.max())

## Step 3 — the payoff rules, at three prices

On expiry day you only use your right if it helps you.

- A **call** lets you buy at ₹1,560. If Reliance is worth more, you gain the difference;
  if not, you walk away with nothing. So the call pays `max(S_T - K, 0)`.
- A **put** lets you sell at ₹1,560. It pays when Reliance ends *below* the strike:
  `max(K - S_T, 0)`.

`np.maximum` keeps the bigger of two numbers, one price at a time. That "or zero" is the
walking away. **Profit** is the payoff minus the premium you paid.

In [ ]:
three_prices = np.array([1450, 1560, 1700])   # three possible prices in a month

call_payoff = np.maximum(three_prices - K, 0)    # right to BUY at 1,560
put_payoff = np.maximum(K - three_prices, 0)     # right to SELL at 1,560

print("Reliance on expiry:", three_prices)
print("call payoff       :", call_payoff)
print("put payoff        :", put_payoff)
print("call profit       :", call_payoff - call_premium)

Check two of these by hand. At ₹1,700 the call pays 1,700 − 1,560 = ₹140, and after
the ₹41 premium the buyer is ₹99 ahead. At ₹1,450 the put pays 1,560 − 1,450 = ₹110.

## Step 4 — draw the call and the put side by side

Now the same rules on all 200 prices. The solid line is the payoff; the dashed line is
the profit, the same shape pushed down by the premium. The bend at the strike is why
people call this shape a **hockey stick**.

In [ ]:
call_payoffs = np.maximum(S_T - K, 0)
put_payoffs = np.maximum(K - S_T, 0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(S_T, call_payoffs, color="#2E75B6", linewidth=3, label="payoff")
axes[0].plot(S_T, call_payoffs - call_premium, color="grey", linestyle="--", label="profit")
axes[0].set_title("Call: the right to BUY at 1,560")

axes[1].plot(S_T, put_payoffs, color="#C0392B", linewidth=3, label="payoff")
axes[1].plot(S_T, put_payoffs - put_premium, color="grey", linestyle="--", label="profit")
axes[1].set_title("Put: the right to SELL at 1,560")

for ax in axes:
    ax.axhline(0, color="grey", linewidth=1)
    ax.axvline(K, color="grey", linestyle=":")
    ax.set_xlabel("Reliance on expiry day (Rs)")
    ax.set_ylabel("Rs per share")
    ax.legend()

plt.tight_layout()
plt.show()

print("Call break-even:", K + call_premium, "  Put break-even:", K - put_premium)

## Step 5 — buyer versus writer

Every option has two sides. The **buyer** pays the premium and gets the right. The
**writer** (the seller) collects the premium and must deliver if the buyer asks. Whatever
the buyer gains, the writer loses, so the writer's profit is just the buyer's profit with
the sign flipped.

In [ ]:
buyer_profit = call_payoffs - call_premium
writer_profit = -buyer_profit

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
axes[0].plot(S_T, buyer_profit, color="#2E7D32", linewidth=3)
axes[0].set_title("Buyer of the call")
axes[1].plot(S_T, writer_profit, color="#C0392B", linewidth=3)
axes[1].set_title("Writer of the call")
for ax in axes:
    ax.axhline(0, color="grey", linewidth=1)
    ax.axvline(K, color="grey", linestyle=":")
    ax.set_xlabel("Reliance on expiry day (Rs)")
axes[0].set_ylabel("profit, Rs per share")
plt.tight_layout()
plt.show()

print("Buyer : worst case", buyer_profit.min(), "  best case on this chart", round(buyer_profit.max()))
print("Writer: best case ", writer_profit.max(), "  worst case on this chart", round(writer_profit.min()))

The buyer can never lose more than ₹41. The writer can never gain more than ₹41, and
the further Reliance rises, the more they lose. "Best case on this chart" is only as far
as our chart goes: a real share has no ceiling.

## Step 6 — in, at or out of the money

**Moneyness** just says whether using the right *today* would pay anything. What it would
pay today is called the **intrinsic value**.

In [ ]:
for price_now in [1650, 1560, 1450, S_today]:
    call_now = max(price_now - K, 0)
    put_now = max(K - price_now, 0)
    print("Reliance at", round(price_now, 2),
          " ->  using the call pays", round(call_now, 2),
          "  using the put pays", round(put_now, 2))

Above the strike, the call is **in the money** and the put is **out of the money**; below
it, the other way round. Today our call is only ₹3 in the money, yet it costs about ₹41.
The other ₹38 is **time value**: the chance Reliance climbs further before expiry.
Notebook 2 puts an exact number on it.

## Step 7 — strategy 1: the protective put (insurance)

Strategies are built by **adding profits together**. Suppose you own one Reliance share,
bought at ₹1,563.17, and you buy the ₹1,560 put. The put is insurance: however far
Reliance falls, you can sell at ₹1,560.

In [ ]:
share_profit = S_T - S_today               # just owning the share
put_profit = put_payoffs - put_premium      # the put we bought
protected = share_profit + put_profit       # share + put

plt.figure(figsize=(8, 5))
plt.plot(S_T, share_profit, color="grey", linestyle="--", label="own the share only")
plt.plot(S_T, put_profit, color="#C0392B", linestyle=":", label="the put only")
plt.plot(S_T, protected, color="#2E7D32", linewidth=3, label="share + put")
plt.axhline(0, color="grey", linewidth=1)
plt.axvline(K, color="grey", linestyle=":")
plt.xlabel("Reliance on expiry day (Rs)")
plt.ylabel("profit, Rs per share")
plt.title("Protective put: a share with insurance")
plt.legend()
plt.show()

print("Worst case with the insurance:", round(protected.min(), 2), "rupees a share")

## Step 8 — strategy 2: the covered call (renting out your shares)

Now you own the share and **sell** the ₹1,560 call. You collect ₹41 now, like rent, but
you have agreed to sell at ₹1,560 if the buyer asks. Selling a call is the writer's side,
so its profit is minus the buyer's.

In [ ]:
sold_call_profit = -(call_payoffs - call_premium)   # we are the writer now
covered = share_profit + sold_call_profit

plt.figure(figsize=(8, 5))
plt.plot(S_T, share_profit, color="grey", linestyle="--", label="own the share only")
plt.plot(S_T, sold_call_profit, color="#2E75B6", linestyle=":", label="the call you sold")
plt.plot(S_T, covered, color="#E08E0B", linewidth=3, label="share + sold call")
plt.axhline(0, color="grey", linewidth=1)
plt.axvline(K, color="grey", linestyle=":")
plt.xlabel("Reliance on expiry day (Rs)")
plt.ylabel("profit, Rs per share")
plt.title("Covered call: collect rent, give up the big rise")
plt.legend()
plt.show()

print("Best case with the covered call:", round(covered.max(), 2), "rupees a share")

## Step 9 — strategy 3: the straddle (a bet on a big move)

Buy the call **and** the put at the same strike. You lose both premiums if Reliance sits
still, but you win on a big move in **either** direction. Think of an election result:
everyone expects a big move, nobody knows which way. On 4 June 2024, results day,
Reliance fell 7.5% in one day.

In [ ]:
straddle = (call_payoffs - call_premium) + (put_payoffs - put_premium)

plt.figure(figsize=(8, 5))
plt.plot(S_T, call_payoffs - call_premium, color="#2E75B6", linestyle=":", label="call only")
plt.plot(S_T, put_payoffs - put_premium, color="#C0392B", linestyle=":", label="put only")
plt.plot(S_T, straddle, color="#6A4C93", linewidth=3, label="straddle: call + put")
plt.axhline(0, color="grey", linewidth=1)
plt.axvline(K, color="grey", linestyle=":")
plt.xlabel("Reliance on expiry day (Rs)")
plt.ylabel("profit, Rs per share")
plt.title("Straddle: win on a big move, either way")
plt.legend()
plt.show()

# What if a results-day fall of 7.5% happened this month?
after_fall = S_today * (1 - 0.075)
profit_after_fall = max(after_fall - K, 0) + max(K - after_fall, 0) - (call_premium + put_premium)
print("Reliance after a 7.5% fall:", round(after_fall, 2))
print("straddle profit           :", round(profit_after_fall, 2), "rupees a share")
print("if Reliance ends at 1,560 :", -(call_premium + put_premium), "rupees a share")

## Step 10 — two portfolios: do they pay the same?

Put-call parity starts with a question, not a formula. Build two portfolios:

- **Portfolio A:** one call, plus cash in a deposit that will be worth ₹1,560 on expiry day.
- **Portfolio B:** one put, plus one Reliance share.

What is each worth on expiry day? Do it at three prices first, then at all 200.

In [ ]:
for price in [1450, 1560, 1700]:
    A = max(price - K, 0) + K       # call payoff + the cash, now grown to 1,560
    B = max(K - price, 0) + price   # put payoff + the share
    print("Reliance ends at", price, " ->  portfolio A:", A, "  portfolio B:", B)

# The same check on every one of our 200 prices.
A_all = call_payoffs + K
B_all = put_payoffs + S_T
print()
print("biggest gap between A and B, over all 200 prices:", np.abs(A_all - B_all).max())

The gap is zero everywhere: both portfolios always end up worth whichever is bigger,
Reliance or ₹1,560. Here is the same fact as a picture.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(S_T, A_all, color="#2E75B6", linewidth=4, label="A: call + cash")
plt.plot(S_T, B_all, color="#C0392B", linewidth=2, linestyle="--", label="B: put + share")
plt.axvline(K, color="grey", linestyle=":")
plt.xlabel("Reliance on expiry day (Rs)")
plt.ylabel("value on expiry day (Rs)")
plt.title("Two portfolios that always pay the same")
plt.legend()
plt.show()

## Step 11 — same payoff, so same price

If two portfolios pay the same in every possible future, they must cost the same **today**.
Otherwise you could buy the cheap one, sell the dear one, and keep free money. That is
**put-call parity**:

$$ C + K e^{-rT} = P + S $$

- Left side: portfolio A today. `C` is the call price, and `K e^(-rT)` is the cash you need
  today so it grows to ₹1,560 in a month at 6% a year.
- Right side: portfolio B today. `P` is the put price and `S` is Reliance's price now.

We check it with the exact prices notebook 2 will compute.

In [ ]:
call_price = 40.81      # from Black-Scholes, notebook 2
put_price = 29.86       # from Black-Scholes, notebook 2
r = 0.06                # interest rate, 6% a year
T = 1 / 12              # one month, in years

cash_today = K * np.exp(-r * T)     # grows to 1,560 in a month

print("cash needed today     :", round(cash_today, 2))
print("portfolio A today C + K*e^(-rT):", round(call_price + cash_today, 2))
print("portfolio B today P + S        :", round(put_price + S_today, 2))

The two sides agree to the paisa. Parity needs no model of how Reliance moves at all,
which makes it the first sanity check for any option-pricing code.

### Stretch (optional) — insurance plus rent is a fixed deposit

Skip this if you are new to finance. Combine strategies 1 and 2: own the share, **buy** the
₹1,560 put and **sell** the ₹1,560 call. Your losses are floored by the put and your gains
are capped by the call, at the same price. What is left?

In [ ]:
# Exact premiums this time.
locked = share_profit + (put_payoffs - put_price) - (call_payoffs - call_price)

print("lowest profit :", round(locked.min(), 2))
print("highest profit:", round(locked.max(), 2))

# What a month of 6% interest earns on the money you put in.
money_in = S_today + put_price - call_price
print("money put in today:", round(money_in, 2),
      "  interest for a month:", round(money_in * (np.exp(r * T) - 1), 2))

The profit is the same whatever Reliance does, and it equals one month's interest on the
money you put in. Share + put − call is a fixed deposit in disguise, which is put-call
parity rearranged: `S + P − C = K e^(-rT)`.

## Your turn

Draw the profit of someone who **sells** (writes) the ₹1,560 put for ₹30. Use `put_payoffs`
and `put_premium`, and remember the writer's profit is minus the buyer's. What is the most
they can gain? Where on the chart do they lose money, and how much could they lose if
Reliance fell to ₹1,350?

In [ ]:
# Your code here: the put writer's profit, a chart, and the two numbers.

## What you just did

You took one real option, the one-month ₹1,560 call and put on Reliance, and drew what it
pays on expiry day: the hockey sticks, the buyer's capped loss and the writer's open risk.
You built insurance (protective put), rent (covered call) and a bet on a big move
(straddle) just by adding profits, and you showed that two different portfolios always
pay the same, so they must cost the same today.

What we have not done yet is say where the ₹40.81 and ₹29.86 come from. That is
notebook 2.